# Week 7 Exercise: Open-Model Return Reason Classifier

In [ ]:
#!uv add transformers datasets accelerate peft trl scikit-learn sentencepiece

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## QLoRA Config

In [ ]:
BASE_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR = 'week7_returns_lora_adapter'
HF_TOKEN = os.getenv('HF_TOKEN', None)

RUN_TRAINING = True
RUN_BASE_EVAL = True
RUN_MODEL_EVAL = True
TRAIN_SLICE = 12000
MANUAL_FALLBACK = False

LABELS = [
    'defective',
    'wrong_item',
    'size_or_fit',
    'late_delivery',
    'not_as_described',
    'changed_mind'
]

JSONL_DIR = 'week7/community_contributions/davidkamere/jsonl'
ARTIFACT_DIR = 'week7/community_contributions/davidkamere/artifacts'
os.makedirs(JSONL_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
USE_QLORA = torch.cuda.is_available()
print('device:', DEVICE, '| use_qlora:', USE_QLORA)
print('jsonl_dir:', JSONL_DIR)
print('artifact_dir:', ARTIFACT_DIR)


## Prompt Data From Online Dataset

In [ ]:
try:
    ds = load_dataset('amazon_reviews_multi', 'en', split=f'train[:{TRAIN_SLICE}]', trust_remote_code=True)
    raw_df = ds.to_pandas()[['review_title', 'review_body', 'stars']].dropna().copy()
    raw_df['text'] = (raw_df['review_title'].fillna('') + '. ' + raw_df['review_body'].fillna('')).str.strip()
except Exception:
    ds = load_dataset('yelp_review_full', split=f'train[:{TRAIN_SLICE}]')
    raw_df = ds.to_pandas()[['text', 'label']].dropna().copy()
    raw_df['stars'] = raw_df['label'].astype(int) + 1

raw_df = raw_df[raw_df['text'].str.len() > 30].reset_index(drop=True)
raw_df.head(3)


In [ ]:
def normalize_text(x: str) -> str:
    x = x.lower().strip()
    x = re.sub(r'\s+', ' ', x)
    return x


def assign_return_reason(text: str, stars: int):
    t = normalize_text(text)
    if any(k in t for k in ['broken', 'defect', 'stopped working', 'damaged', 'does not work', "didn't work"]):
        return 'defective'
    if any(k in t for k in ['wrong item', 'wrong product', 'received different', 'not what i ordered', 'different item']):
        return 'wrong_item'
    if any(k in t for k in ['too small', 'too big', 'does not fit', "didn't fit", 'size was off']):
        return 'size_or_fit'
    if any(k in t for k in ['arrived late', 'late delivery', 'delayed shipping', 'shipping took too long', 'expected date']):
        return 'late_delivery'
    if any(k in t for k in ['not as described', 'misleading', 'looked different', 'description was wrong', 'nothing like the photo']):
        return 'not_as_described'
    if stars <= 2 and any(k in t for k in ['returned', 'returning', 'refund', 'not worth it', 'changed my mind']):
        return 'changed_mind'
    return None


def keyword_predict(text: str) -> str:
    t = normalize_text(text)
    if any(k in t for k in ['broken', 'defect', 'stopped working', 'damaged', 'does not work', "didn't work"]):
        return 'defective'
    if any(k in t for k in ['wrong item', 'wrong product', 'received different', 'not what i ordered', 'different item']):
        return 'wrong_item'
    if any(k in t for k in ['too small', 'too big', 'does not fit', "didn't fit", 'size was off']):
        return 'size_or_fit'
    if any(k in t for k in ['arrived late', 'late delivery', 'delayed shipping', 'shipping took too long', 'expected date']):
        return 'late_delivery'
    if any(k in t for k in ['not as described', 'misleading', 'looked different', 'description was wrong', 'nothing like the photo']):
        return 'not_as_described'
    if any(k in t for k in ['returned', 'returning', 'refund', 'not worth it', 'changed my mind']):
        return 'changed_mind'
    return 'changed_mind'


df = raw_df.copy()
df['label'] = [assign_return_reason(t, int(s)) for t, s in zip(df['text'], df['stars'])]
df = df[df['label'].isin(LABELS)].copy().reset_index(drop=True)

counts = df['label'].value_counts()
df = df[df['label'].isin(counts[counts >= 3].index)].copy().reset_index(drop=True)

print('rows:', len(df))
print(df['label'].value_counts())


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['label'])
if temp_df['label'].value_counts().min() >= 2:
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])
else:
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))


## Export Week 7 JSONL\n

In [ ]:
SYSTEM_PROMPT_JSONL = globals().get(
    'SYSTEM_PROMPT',
    'You classify customer return reasons. Return exactly one label from: defective, wrong_item, size_or_fit, late_delivery, not_as_described, changed_mind.'
)

def training_record(row):
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT_JSONL},
            {'role': 'user', 'content': row['text']},
            {'role': 'assistant', 'content': row['label']}
        ]
    }

def write_jsonl(frame, path):
    import json
    with open(path, 'w') as f:
        for _, r in frame.iterrows():
            f.write(json.dumps(training_record(r), ensure_ascii=False) + '\n')

train_jsonl = os.path.join(JSONL_DIR, 'week7_returns_train.jsonl')
val_jsonl = os.path.join(JSONL_DIR, 'week7_returns_val.jsonl')

write_jsonl(train_df, train_jsonl)
write_jsonl(val_df, val_jsonl)

print('wrote:', train_jsonl, 'rows=', len(train_df))
print('wrote:', val_jsonl, 'rows=', len(val_df))


In [ ]:
if RUN_BASE_EVAL:
    baseline_preds = [keyword_predict(t) for t in test_df['text'].tolist()]
    truth = test_df['label'].tolist()

    print('Baseline accuracy:', round(accuracy_score(truth, baseline_preds), 4))
    print('Baseline macro_f1:', round(f1_score(truth, baseline_preds, average='macro'), 4))
else:
    print('Set RUN_BASE_EVAL=True to run baseline metrics')


## Prompt Formatting + Tokenization Check

In [ ]:
SYSTEM_PROMPT = (
    'You classify customer return reasons. '
    'Return exactly one label from: defective, wrong_item, size_or_fit, late_delivery, not_as_described, changed_mind.'
)


def make_prompt(text: str) -> str:
    return (
        '<|system|>\n' + SYSTEM_PROMPT + '\n'
        '<|user|>\n' + text + '\n'
        '<|assistant|>\n'
    )


def to_sft(frame: pd.DataFrame):
    out = frame[['text', 'label']].copy()
    out['text'] = [make_prompt(t) + l for t, l in zip(out['text'], out['label'])]
    return Dataset.from_pandas(out[['text']], preserve_index=False)

train_sft = to_sft(train_df)
val_sft = to_sft(val_df)
print(train_sft[0]['text'][:300])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample_lengths = [len(tokenizer(x['text'], truncation=True, max_length=512)['input_ids']) for x in train_sft.select(range(min(200, len(train_sft))))]
print('token length p50/p95/max:', int(np.percentile(sample_lengths, 50)), int(np.percentile(sample_lengths, 95)), int(np.max(sample_lengths)))


## Fine-Tuning Setup

In [ ]:
dtype = torch.float16 if DEVICE in ['cuda', 'mps'] else torch.float32
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, torch_dtype=dtype)
if DEVICE != 'cpu':
    base_model = base_model.to(DEVICE)
base_model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=100,
    fp16=(DEVICE == 'cuda'),
    bf16=False,
    report_to='none'
)

trainer = SFTTrainer(
    model=base_model,
    args=sft_args,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    peft_config=peft_config,
    processing_class=tokenizer
)

if RUN_TRAINING:
    trainer.train()
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print('saved adapter to', OUTPUT_DIR)
else:
    print('RUN_TRAINING=False, skipping train')


In [ ]:
adapter_config = os.path.join(OUTPUT_DIR, 'adapter_config.json')
if os.path.isfile(adapter_config):
    inference_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    using_adapter = True
else:
    inference_model = base_model
    using_adapter = False

inference_model.eval()
print('using_adapter:', using_adapter)


## Evaluation

In [ ]:
def score_label(text: str, label: str) -> float:
    prompt = make_prompt(text)
    full = prompt + label

    tok_full = tokenizer(full, return_tensors='pt', truncation=True, max_length=512)
    tok_prompt = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)

    input_ids = tok_full['input_ids'].to(inference_model.device)
    attention_mask = tok_full['attention_mask'].to(inference_model.device)

    labels = input_ids.clone()
    prompt_len = tok_prompt['input_ids'].shape[1]
    labels[:, :prompt_len] = -100

    with torch.no_grad():
        out = inference_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    return float(out.loss.item())


def llm_predict(text: str):
    scored = [(lbl, score_label(text, lbl)) for lbl in LABELS]
    scored.sort(key=lambda x: x[1])
    pred = scored[0][0]
    if pred is None and MANUAL_FALLBACK:
        pred = keyword_predict(text)
    return pred if pred is not None else 'unknown'


## Balanced Manual Evaluation Set


In [ ]:
manual_eval_df = pd.DataFrame([
    {'text': 'The product arrived cracked and unusable straight out of the box.', 'label': 'defective'},
    {'text': 'It worked for one day, then completely died.', 'label': 'defective'},
    {'text': 'Button stopped functioning after first use.', 'label': 'defective'},
    {'text': 'The screen has dead pixels and flickers nonstop.', 'label': 'defective'},

    {'text': 'I got a different color and model than what I purchased.', 'label': 'wrong_item'},
    {'text': 'The box had another product entirely.', 'label': 'wrong_item'},
    {'text': 'This is not the item listed in my order.', 'label': 'wrong_item'},
    {'text': 'They shipped the wrong variant to me.', 'label': 'wrong_item'},

    {'text': 'These pants are much tighter than expected.', 'label': 'size_or_fit'},
    {'text': 'The jacket is oversized and hangs awkwardly.', 'label': 'size_or_fit'},
    {'text': 'Sizing chart was off and it does not fit properly.', 'label': 'size_or_fit'},
    {'text': 'I cannot wear it because the fit is completely wrong.', 'label': 'size_or_fit'},

    {'text': 'Shipping took far longer than promised.', 'label': 'late_delivery'},
    {'text': 'It arrived several days past the delivery window.', 'label': 'late_delivery'},
    {'text': 'My package only came after multiple delays.', 'label': 'late_delivery'},
    {'text': 'Delivery missed the expected date by a week.', 'label': 'late_delivery'},

    {'text': 'What I got does not match the listing details.', 'label': 'not_as_described'},
    {'text': 'The item quality and features differ from the description.', 'label': 'not_as_described'},
    {'text': 'Photos and written specs were misleading.', 'label': 'not_as_described'},
    {'text': 'It looked and felt very different from what was advertised.', 'label': 'not_as_described'},

    {'text': 'I decided I do not need this anymore and want to return it.', 'label': 'changed_mind'},
    {'text': 'Nothing is wrong, I just no longer want it.', 'label': 'changed_mind'},
    {'text': 'I purchased in a hurry and now prefer to send it back.', 'label': 'changed_mind'},
    {'text': 'I want a return for personal preference reasons.', 'label': 'changed_mind'}
])
manual_eval_df['label'].value_counts()


In [ ]:
if RUN_MODEL_EVAL:
    truth_test = test_df['label'].tolist()
    llm_preds_test = [llm_predict(t) for t in test_df['text'].tolist()]

    print('LLM/Adapter Test accuracy:', round(accuracy_score(truth_test, llm_preds_test), 4))
    print('LLM/Adapter Test macro_f1:', round(f1_score(truth_test, llm_preds_test, average='macro'), 4))

    truth_manual = manual_eval_df['label'].tolist()
    baseline_manual = [keyword_predict(t) for t in manual_eval_df['text'].tolist()]
    llm_manual = [llm_predict(t) for t in manual_eval_df['text'].tolist()]

    print('Manual baseline accuracy:', round(accuracy_score(truth_manual, baseline_manual), 4))
    print('Manual baseline macro_f1:', round(f1_score(truth_manual, baseline_manual, average='macro'), 4))
    print('Manual model accuracy:', round(accuracy_score(truth_manual, llm_manual), 4))
    print('Manual model macro_f1:', round(f1_score(truth_manual, llm_manual, average='macro'), 4))
    print(classification_report(truth_manual, llm_manual, digits=4, zero_division=0))

    cm_manual = pd.DataFrame(confusion_matrix(truth_manual, llm_manual, labels=LABELS), index=LABELS, columns=LABELS)
    display(cm_manual)

    results_rows = [
        {
            'eval_set': 'manual_balanced',
            'model_variant': 'baseline_keyword',
            'accuracy': round(accuracy_score(truth_manual, baseline_manual), 4),
            'macro_f1': round(f1_score(truth_manual, baseline_manual, average='macro'), 4)
        },
        {
            'eval_set': 'manual_balanced',
            'model_variant': 'week7_model',
            'accuracy': round(accuracy_score(truth_manual, llm_manual), 4),
            'macro_f1': round(f1_score(truth_manual, llm_manual, average='macro'), 4)
        }
    ]

    results_df = pd.DataFrame(results_rows)
    results_path = os.path.join(ARTIFACT_DIR, 'week7_results.csv')
    eval_path = os.path.join(ARTIFACT_DIR, 'week7_manual_eval_set.csv')
    results_df.to_csv(results_path, index=False)
    manual_eval_df.to_csv(eval_path, index=False)
    print('saved:', results_path)
    print('saved:', eval_path)
    display(results_df)
else:
    print('Set RUN_MODEL_EVAL=True to run model evaluation')


## Standalone File Check\n

In [ ]:
expected = [
    os.path.join(JSONL_DIR, 'week7_returns_train.jsonl'),
    os.path.join(JSONL_DIR, 'week7_returns_val.jsonl'),
    os.path.join(ARTIFACT_DIR, 'week7_results.csv'),
    os.path.join(ARTIFACT_DIR, 'week7_eval_set.csv')
]
for p in expected:
    print(p, '=>', os.path.exists(p))


## Demo

In [ ]:
demo = [
    'I received a different item than what I ordered and want a refund.',
    'The shoes are too small and do not fit me.',
    'The package arrived very late after the expected date.',
    'The blender stopped working after one day.',
    'The product looked nothing like the photo in the listing.',
    'I changed my mind and want to return it.'
]

for i, text in enumerate(demo, 1):
    print(f'{i}. {text}')
    print('baseline ->', keyword_predict(text))
    print('model    ->', llm_predict(text))
    print('-' * 80)
